##### Part 1 Data Extraction

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd

from tqdm import tqdm
from datetime import datetime

In [2]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META"]

In [3]:
yf.Ticker('APPL').info

{'quoteType': 'MUTUALFUND',
 'symbol': 'APPL',
 'language': 'en-US',
 'region': 'US',
 'typeDisp': 'Fund',
 'quoteSourceName': 'Delayed Quote',
 'triggerable': False,
 'customPriceAlertConfidence': 'NONE',
 'corporateActions': [],
 'regularMarketTime': 1561759658,
 'exchange': 'YHD',
 'exchangeTimezoneName': 'America/New_York',
 'exchangeTimezoneShortName': 'EDT',
 'gmtOffSetMilliseconds': -14400000,
 'market': 'us_market',
 'esgPopulated': False,
 'marketState': 'CLOSED',
 'hasPrePostMarketData': False,
 'priceHint': 2,
 'fullExchangeName': 'YHD',
 'sourceInterval': 15,
 'exchangeDataDelayedBy': 0,
 'tradeable': False,
 'cryptoTradeable': False,
 'trailingPegRatio': None}

In [4]:
def get_company_info(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    return {
        'ticker': ticker,
        'company_name': info.get('longName'),
        'sector': info.get('sector'),
        'industry': info.get('industry'),
        'country': info.get('country'),
        'market_cap': info.get('marketCap'),
        'website': info.get('website')
    }

get_company_info('NVDA')

{'ticker': 'NVDA',
 'company_name': 'NVIDIA Corporation',
 'sector': 'Technology',
 'industry': 'Semiconductors',
 'country': 'United States',
 'market_cap': 4967727366144,
 'website': 'https://www.nvidia.com'}

In [5]:
company_records = []

for ticker in tickers:
    try:
        company_records.append(get_company_info(ticker))
    except Exception as e:
        print(ticker, e)

companies_df = pd.DataFrame(company_records)
companies_df.head()

,ticker,company_name,sector,industry,country,market_cap,website
0,AAPL,Apple Inc.,Technology,Consumer Electronics,United States,4514011676672,https://www.apple.com
1,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,United States,3095206035456,https://www.microsoft.com
2,NVDA,NVIDIA Corporation,Technology,Semiconductors,United States,4967727366144,https://www.nvidia.com
3,AMZN,"Amazon.com, Inc.",Consumer Cyclical,Internet Retail,United States,2646571745280,https://www.amazon.com
4,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,United States,4494199357440,https://abc.xyz


In [6]:
companies_df.to_parquet("companies.parque", index = False)

In [7]:
def get_price_history(ticker, period = '5y'):
    stock = yf.Ticker(ticker)
    df = stock.history(period = period)
    df.reset_index(inplace = True)
    df['ticker'] = ticker
    return df

nvda_prices = get_price_history('NVDA')
nvda_prices

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,ticker
0,2021-06-07 00:00:00-04:00,17.506672,17.747034,17.129563,17.554245,575756000,0.000,0.0,NVDA
1,2021-06-08 00:00:00-04:00,17.462833,17.556736,17.187597,17.392841,323848000,0.000,0.0,NVDA
2,2021-06-09 00:00:00-04:00,17.455376,17.514422,17.196272,17.298418,381656000,0.004,0.0,NVDA
3,2021-06-10 00:00:00-04:00,17.290200,17.431711,17.116800,17.364941,287772000,0.000,0.0,NVDA
4,2021-06-11 00:00:00-04:00,17.419246,17.877413,17.383620,17.763805,416308000,0.000,0.0,NVDA
...,...,...,...,...,...,...,...,...,...
1251,2026-06-01 00:00:00-04:00,215.478858,224.608217,215.448894,224.098816,212850700,0.000,0.0,NVDA
1252,2026-06-02 00:00:00-04:00,226.915518,232.009586,221.092318,222.560608,193362900,0.000,0.0,NVDA
1253,2026-06-03 00:00:00-04:00,221.461887,222.560613,214.260274,214.500000,160907000,0.000,0.0,NVDA
1254,2026-06-04 00:00:00-04:00,213.910004,221.600006,210.970001,218.660004,169022200,0.250,0.0,NVDA


In [8]:
price_frames = []
for ticker in tqdm(tickers):
    try:
        df = get_price_history(ticker)
        price_frames.append(df)
    except Exception as e:
        print(ticker, e)

prices_df = pd.concat(price_frames)
print(prices_df.shape)

100%|██████████| 6/6 [00:00<00:00,  6.54it/s]

(7536, 9)


In [9]:

prices_df.to_parquet("prices.parquet", index = False)


In [10]:
def get_income_statement(ticker):
    stock = yf.Ticker(ticker)
    income = stock.financials.T
    income['ticker'] = ticker
    return income

def get_balance_sheet(ticker):
    stock = yf.Ticker(ticker)
    balance_sheet = stock.balance_sheet.T
    balance_sheet['ticker'] = ticker
    return balance_sheet

def get_cashflow(ticker):
    stock = yf.Ticker(ticker)
    cashflow = stock.cashflow.T
    cashflow['ticker'] = ticker
    return cashflow


In [11]:
income_frames = []
balance_frames = []
cashflow_frames = []

for ticker in tqdm(tickers):
    try:
        income_frames.append(get_income_statement(ticker))
        balance_frames.append(get_balance_sheet(ticker))
        cashflow_frames.append(get_cashflow(ticker))
    except Exception as e:
        print(ticker, e)


income_df = pd.concat(income_frames)
balance_df = pd.concat(balance_frames)
cashflow_df = pd.concat(cashflow_frames)

income_df.to_parquet("income.parquet", index=False)
balance_df.to_parquet("balance.parquet", index=False)
cashflow_df.to_parquet("cashflow.parquet", index=False)

100%|██████████| 6/6 [00:04<00:00,  1.30it/s]


In [19]:
income_df.isnull().mean().sort_values()

ticker                                                        0.000000
Tax Effect Of Unusual Items                                   0.111111
Net Income Common Stockholders                                0.111111
Net Income                                                    0.111111
Net Income Including Noncontrolling Interests                 0.111111
Net Income Continuous Operations                              0.111111
Tax Provision                                                 0.111111
Pretax Income                                                 0.111111
Other Non Operating Income Expenses                           0.111111
Operating Income                                              0.111111
Operating Expense                                             0.111111
Selling General And Administration                            0.111111
Gross Profit                                                  0.111111
Cost Of Revenue                                               0.111111
Total 

In [20]:
prices_df.duplicated().sum()

np.int64(0)

In [21]:
prices_df["ticker"].nunique()

6

##### Part 2: News Pipeline


The logic is based on a 4-layer pipeline to process the news and output a formatted message.

Raw News -> Sentiment Layer -> Event Layer -> Risk Layer -> Entity Layer -> Feature Store

In [12]:
from transformers import pipeline

/Users/thomasxiang/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
ticker = yf.Ticker('NVDA')
ticker.news

[{'id': 'dc6d5fc5-bb0c-4fdc-99f5-309ec954b1b5',
  'content': {'id': 'dc6d5fc5-bb0c-4fdc-99f5-309ec954b1b5',
   'contentType': 'STORY',
   'title': 'AI stock mania is taking over the markets in 2026',
   'description': '',
   'summary': 'It was a mind-blowing week for the AI trade, with only one reasonable conclusion.',
   'pubDate': '2026-06-07T12:30:00Z',
   'displayTime': '2026-06-07T12:30:00Z',
   'isHosted': True,
   'bypassModal': False,
   'previewUrl': None,
   'thumbnail': {'originalUrl': 'https://s.yimg.com/uu/api/res/1.2/iu8qDSZOEooI60uI6uHNag--~B/aD0zOTk5O3c9NjAwMDthcHBpZD15dGFjaHlvbg--/https://d29szjachogqwa.cloudfront.net/images/2026-06/7806654d-63ff-4cfb-8f0a-2fe1ac08c92c',
    'originalWidth': 6000,
    'originalHeight': 3999,
    'caption': '',
    'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/iu8qDSZOEooI60uI6uHNag--~B/aD0zOTk5O3c9NjAwMDthcHBpZD15dGFjaHlvbg--/https://d29szjachogqwa.cloudfront.net/images/2026-06/7806654d-63ff-4cfb-8f0a-2fe1ac08c92c',
      

In [28]:
ticker.news[0]['content'].keys()

dict_keys(['id', 'contentType', 'title', 'description', 'summary', 'pubDate', 'displayTime', 'isHosted', 'bypassModal', 'previewUrl', 'thumbnail', 'provider', 'canonicalUrl', 'clickThroughUrl', 'metadata', 'finance', 'storyline'])

In [ ]:
def build_article_text(article):
    return f"""
    Title:
    {article.get('title', '')}

    Description:
    {article.get('description', '')}

    Summary:
    {article.get('summary', '')}

    
    """

In [ ]:
sentiment_model = pipeline(
    "text-classification", 
    model = "ProsusAI/finbert", 
    top_k = None)


text = build_article_text(article=ticker.news[0]['content'])


result = sentiment_model(text[:512])

print(result)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 94449.37it/s]


[[{'label': 'neutral', 'score': 0.8348400592803955}, {'label': 'negative', 'score': 0.09783732146024704}, {'label': 'positive', 'score': 0.06732263416051865}]]


In [50]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

sentiment_model = pipeline(
    task = "text-classification", 
    model = model,
    tokenizer = tokenizer,
    top_k = None)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 61406.88it/s]


In [62]:
tokens = tokenizer.tokenize(build_article_text(article=ticker.news[9]['content']))
print(tokens[:20])
print(len(tokens))

['title', ':', '1', 'costly', 'mistake', 'too', 'many', 'investors', 'make', 'with', 'the', 'vanguard', 's', '&', 'p', '500', 'et', '##f', '(', 'vo']
44


In [49]:
sentiment_model(
    text,
    truncation=True,
    max_length=512
)

[[{'label': 'neutral', 'score': 0.8348400592803955},
  {'label': 'negative', 'score': 0.09783732146024704},
  {'label': 'positive', 'score': 0.06732263416051865}]]

In [55]:
result[0]

[{'label': 'neutral', 'score': 0.8348400592803955},
 {'label': 'negative', 'score': 0.09783732146024704},
 {'label': 'positive', 'score': 0.06732263416051865}]

In [56]:
def get_finbert_sentiment(text: str) -> dict:


    results = sentiment_model(
        text,
        truncation=True,
        max_length=512
    )[0]

    scores = {
        item["label"].lower(): float(item["score"])
        for item in results
    }

    sentiment = max(
        scores,
        key=scores.get
    )

    return {
        "sentiment": sentiment,
        "sentiment_score": scores[sentiment],
        "positive_score": scores.get("positive", 0),
        "negative_score": scores.get("negative", 0),
        "neutral_score": scores.get("neutral", 0)
    }

In [57]:
article_text = build_article_text(
    ticker.news[0]["content"]
)

sentiment = get_finbert_sentiment(
    article_text
)

print(sentiment)

{'sentiment': 'neutral', 'sentiment_score': 0.8348400592803955, 'positive_score': 0.06732263416051865, 'negative_score': 0.09783732146024704, 'neutral_score': 0.8348400592803955}


In [63]:
def chunk_text_by_tokens(
    text: str,
    tokenizer,
    max_tokens: int = 512,
    overlap: int = 50
):

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(token_ids):

        end = start + max_tokens

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

        start += max_tokens - overlap

    return chunks

In [64]:
long_article = """
Title: MU, NVDA, ORCL, MRVL: Why Are AI and Chip Stocks Bleeding Red Today?

Description:
Chip and AI stocks are under pressure today due to weak signals from the sector and hotter-than-expected economic data. On top of that, traders are taking profits in some of the market’s biggest winners, adding fuel to the selloff. Micron MU -13.25% ▼ , Nvidia NVDA -6.20% ▼ , Oracle ORCL -9.59% ▼ , and Marvell MRVL -16.74% ▼ have declined 11.5%, 7%, 10%, and 13%, respectively, as investors reassess how fast AI spending can grow and how long high interest rates may stay high.



Claim 55% Off TipRanks

Explore NVDS for 2X short leverage on NVDA


Key reasons behind the market’s sharp decline are:


1. Broadcom’s “Reality Check” on AI Spending: The drop began after Broadcom’s AVGO -7.92% ▼ earnings report on June 3. While the company beat estimates, it did not raise its long‑term AI revenue outlook for Fiscal 2026 and 2027. This hit the semiconductor market hard, as a flat forecast signaled that demand may not be rising as quickly as hoped.


2. Hot Jobs Data Pushes Yields Higher: The strong May jobs report showed hiring was hotter than expected, pushing Treasury yields higher and raising worries about a Fed rate hike. Chip and AI stocks are sensitive to rising rates as their biggest profits are expected far in the future. When yields rise, those future earnings get discounted more, and stock prices fall quickly.


3. Heavy Profit‑Taking After a Massive Rally: Many chip names were already stretched after a long run fueled by AI optimism. With valuations sitting at record highs and technical indicators flashing overbought, traders were ready to take profits at the first sign of weakness.


Which Is a Better Stock, MU, NVDA, ORCL, or MRVL?

Using TipRanks’ Stock Comparison Tool, we compared MU, NVDA, ORCL, and MRVL to see which AI stock Wall Street currently favors. All the stocks have a Strong Buy consensus rating on TipRanks, but analysts see the most upside potential in Nvidia stock. The average price target of $311.41 implies about 51.82% upside from the current level.


Summary: 
Chip and AI stocks are sliding after weak forward guidance from Broadcom.
Hot May jobs data pushed yields higher and revived Fed rate‑hike fears.
MU, NVDA, ORCL, and MRVL dropped 11.5%, 7%, 10%, and 13%, respectively.

"""

In [65]:
chunks = chunk_text_by_tokens(
    long_article,
    tokenizer
)

print(len(chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (567 > 512). Running this sequence through the model will result in indexing errors


2


In [66]:
def sentiment_from_long_article(
    text: str,
    tokenizer,
    sentiment_model,
    max_tokens: int = 512,
    overlap: int = 50
):

    chunks = chunk_text_by_tokens(
        text=text,
        tokenizer=tokenizer,
        max_tokens=max_tokens,
        overlap=overlap
    )

    chunk_results = []

    for chunk in chunks:

        result = get_finbert_sentiment(chunk)

        chunk_results.append(result)

    positive = sum(
        r["positive_score"]
        for r in chunk_results
    ) / len(chunk_results)

    negative = sum(
        r["negative_score"]
        for r in chunk_results
    ) / len(chunk_results)

    neutral = sum(
        r["neutral_score"]
        for r in chunk_results
    ) / len(chunk_results)

    final_scores = {
        "positive": positive,
        "negative": negative,
        "neutral": neutral
    }

    final_sentiment = max(
        final_scores,
        key=final_scores.get
    )

    return {
        "sentiment": final_sentiment,
        "sentiment_score": final_scores[final_sentiment],
        "positive_score": positive,
        "negative_score": negative,
        "neutral_score": neutral,
        "num_chunks": len(chunk_results),
        "chunk_results": chunk_results
    }

In [67]:
result = sentiment_from_long_article(
    text=long_article,
    tokenizer=tokenizer,
    sentiment_model=sentiment_model
)

print(result)

{'sentiment': 'negative', 'sentiment_score': 0.9478254616260529, 'positive_score': 0.0224152822047472, 'negative_score': 0.9478254616260529, 'neutral_score': 0.02975929155945778, 'num_chunks': 2, 'chunk_results': [{'sentiment': 'negative', 'sentiment_score': 0.9223329424858093, 'positive_score': 0.03486869856715202, 'negative_score': 0.9223329424858093, 'neutral_score': 0.04279840365052223}, {'sentiment': 'negative', 'sentiment_score': 0.9733179807662964, 'positive_score': 0.009961865842342377, 'negative_score': 0.9733179807662964, 'neutral_score': 0.016720179468393326}]}


Imrpovement:

#TODO:

Instead of using simple average, the new sentiment score can be reweighted by different componments of the news.

For example:
Headline Weight - 40%, Description Weight - 30%, Summary Weight - 10%, Body Weight - 10%

These weights would be a part of tuning process.


##### Event Layer && Risk Layer && Entity Layer

In [71]:
DEEPSEEK_API_KEY = "sk-78de867c66fc4ef3b6b001b6298dcec2"

In [ ]:
from openai import OpenAI

llm_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)



In [96]:
EVENT_KEYWORDS = {
    "earnings_beat": [
        "beat estimates",
        "beats estimates",
        "beats expectations",
        "strong earnings"
    ],

    "earnings_miss": [
        "missed estimates",
        "missed expectations"
    ],

    "guidance_raise": [
        "raised guidance",
        "raises guidance",
        "increased outlook"
    ],

    "guidance_cut": [
        "cut guidance",
        "lowered outlook"
    ],

    "acquisition": [
        "acquired",
        "acquires",
        "acquisition"
    ],

    "partnership": [
        "partnership",
        "partnered",
        "collaboration"
    ],

    "lawsuit": [
        "lawsuit",
        "sued",
        "litigation"
    ],

    "stock_buyback": [
        "buyback",
        "share repurchase"
    ]
}

RISK_KEYWORDS = {
    "competition": [
        "competition",
        "competitive pressure",
        "rival"
    ],

    "regulatory": [
        "regulatory",
        "investigation",
        "sec"
    ],

    "litigation": [
        "lawsuit",
        "legal action",
        "litigation"
    ],

    "valuation": [
        "overvalued",
        "valuation concern"
    ],

    "supply_chain": [
        "supply chain",
        "shortage"
    ],

    "cybersecurity": [
        "cyber attack",
        "security breach"
    ]
}

In [97]:
def rule_based_events(text):

    text = text.lower()

    events = []

    for event, keywords in EVENT_KEYWORDS.items():

        if any(
            keyword in text
            for keyword in keywords
        ):
            events.append(event)

    return list(set(events))


def rule_based_risks(text):

    text = text.lower()

    risks = []

    for risk, keywords in RISK_KEYWORDS.items():

        if any(
            keyword in text
            for keyword in keywords
        ):
            risks.append(risk)

    return list(set(risks))

In [98]:
from pydantic import BaseModel
from typing import List
import json
import re

def extract_json(text):

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if not match:
        return {}

    try:
        return json.loads(
            match.group()
        )
    except:
        return {}


In [99]:
def llm_extract_news(
    article_text,
    client
):

    prompt = f"""
Analyze this financial article.

Return ONLY JSON.

{{
  "events": [],
  "risks": [],
  "entities": []
}}

Article:

{article_text}
"""

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )

    return extract_json(
        response.choices[0].message.content
    )

In [100]:
def extract_news_intelligence(
    article_text,
    client
):

    sentiment = get_finbert_sentiment(
        article_text
    )

    llm_output = llm_extract_news(
        article_text,
        client
    )

    llm_events = llm_output.get(
        "events",
        []
    )

    llm_risks = llm_output.get(
        "risks",
        []
    )

    llm_entities = llm_output.get(
        "entities",
        []
    )

    rule_events = rule_based_events(
        article_text
    )

    rule_risks = rule_based_risks(
        article_text
    )

    final_events = sorted(
        list(
            set(
                llm_events +
                rule_events
            )
        )
    )

    final_risks = sorted(
        list(
            set(
                llm_risks +
                rule_risks
            )
        )
    )

    return {
        **sentiment,

        "events": final_events,

        "risks": final_risks,

        "entities": llm_entities
    }

In [101]:
article_text = build_article_text(
    ticker.news[0]["content"]
)

result = extract_news_intelligence(
    article_text,
    llm_client
)

print(json.dumps(
    result,
    indent=2
))

{
  "sentiment": "neutral",
  "sentiment_score": 0.8348400592803955,
  "positive_score": 0.06732263416051865,
  "negative_score": 0.09783732146024704,
  "neutral_score": 0.8348400592803955,
  "events": [
    "AI stock mania taking over markets in 2026"
  ],
  "risks": [
    "Potential market overvaluation or bubble in AI stocks"
  ],
  "entities": [
    "AI trade"
  ]
}


The result shows 

```
  {
    "sentiment": "neutral",
    "sentiment_score": 0.8348400592803955,
    "positive_score": 0.06732263416051865,
    "negative_score": 0.09783732146024704,
    "neutral_score": 0.8348400592803955,
    "events": [
      "AI stock mania taking over markets in 2026"
    ],
    "risks": [
      "Potential market overvaluation or bubble in AI stocks"
    ],
    "entities": [
      "AI trade"
    ]
  }
  ```

It shows bad news feed and AI hullucination can lead to poor results. It can be also led by other plausible cause.

In [102]:
# Add a validation layer

def validate_output(result):
    
    # remove hallucinated long phrases
    result["events"] = [
        e for e in result["events"]
        if len(e.split()) <= 5
    ]
    
    result["risks"] = [
        r for r in result["risks"]
        if len(r.split()) <= 6
    ]
    
    return result


In [ ]:
from newsapi import NewsApiClient

# Init
newsapi = NewsApiClient(api_key='b08a728c809d44c5acabf818940a0f84')

/Users/thomasxiang/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
